<a href="https://colab.research.google.com/github/VivianCloud/BWSI-GirlsWhoCanLearn-Projects/blob/main/DocumentSkimmer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [79]:
!pip install pymupdf # used so python can read pdf files

In [80]:
from google.colab import files # so you can upload files
import fitz

# these are used to clean the data --- how??
import re
import unicodedata

from sklearn.feature_extraction.text import TfidfVectorizer # this module lets you convert text into vectors
from sklearn.metrics.pairwise import cosine_similarity # this lets you compare how similar vectors are based on angle between them

In [81]:
uploaded = files.upload() # similar to input(), this asks user for a file

Saving stuyAcademicPolicies.pdf to stuyAcademicPolicies (4).pdf


In [82]:
# create the records list variable
records = [];

In [83]:
# document is created from the file
for filename in uploaded.keys():
  document = fitz.open(filename)

In [84]:
def clean_text(text):
  text = unicodedata.normalize("NFKC", text) # normalizes unusual unicode characters
  text = text.replace("\u200b", "") # removes invisible characters
  text = text.replace("\xa0", " ") # replace weird spaces with normal ones
  text = text.replace("\u202f", " ") # same
  text = re.sub(r"\s+", " ", text) # replaces blocks of any whitespace character with one regular space. removes paragraph formatting

  return text.lower()

In [85]:
# a dictionary set manually of the section titles and the pages they begin
SECTIONS = {
    1: "Instructional Expectations",
    4: "Grading Framework",
    7: "Explanation of Grades",
    10: "School Programming - Scope and Sequence",
    11: "Homework Policy",
    13: "The Spiral of Communication",
    14: "Testing/Assessment Policy",
    16: "Academic Integrity Policy",
    18: "Accomodation Room(s) Policy"
}

In [86]:
section_starts = list(SECTIONS.keys()) # gets a list of the keys (the pages) of the sections

for i, start_page in enumerate(section_starts):

  # find the end of the section
  if i + 1 < len(section_starts):
    end_page = section_starts[i + 1] # finds the end as the start of the next section
  else: # this is if i is on the last section
    end_page = len(document)

  # extract text from all pages in this section
  section_text = ""
  for page_num in range(start_page, end_page):
    page_text = document[page_num]
    section_text += page_text.get_text() + " "

  section_text = clean_text(section_text) # cleans the text we got

  records.append((SECTIONS[start_page], section_text, "")) # add the section to records

In [87]:
for reference, text, tags in records:
  print(reference)
  print(f"Length {len(text)}")
  print(f"Preview: {text[:300]}")
  print("-----")

Instructional Expectations
Length 8034
Preview:  new york city public schools (nycps) stuyvesant high school 345 chambers street new york, ny 10282 instructional expectations school year 2026 – 2027 stuyvesant high school maintains a set of instructional expectations to promote learning, accountability, and academic integrity; to minimize stress 
-----
Grading Framework
Length 6465
Preview:  new york city public schools (nycps) stuyvesant high school 345 chambers street new york, ny 10282 grading framework school year 2026 – 2027 grades are an important way of communicating with our students; they provide feedback to them (and families) on their learning progress and their acquisition 
-----
Explanation of Grades
Length 4755
Preview:  new york city public schools (nycps) stuyvesant high school 345 chambers street new york, ny 10282 explanation of grades school year 2026 – 2027 grades (also referred as course marks) reflect students’ understanding and command of content, their progressi

In [88]:
# search_text is the thing we actually search for. It is a list of strings
search_text = [text + " " for pageRef, text, keyWords in records]

In [89]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2)) # stop_words removes the common words such as "the" and "a"
document_vectors = vectorizer.fit_transform(search_text) # document_vectors is a 2D matrix, though fit_transform() returns a sparse matrix

In [90]:
def research(question, top_k=3, min_similarity=0.001):
    query_vector = vectorizer.transform([question])
    scores = cosine_similarity(query_vector, document_vectors).ravel() # then first one is similarity with first doc, second with second doc, and so forth

    # testing the shapes
    print(document_vectors.shape) # should be num of articles x num of vocab features
    print(query_vector.shape) # 1 x num vocab features
    print(cosine_similarity(query_vector, document_vectors).shape) # 1 x num articles
    print(scores.shape) # 9 x --> then gets flattened by .ravel() to just be 1D list

    print("QUESTION:", question)
    if scores.max() < min_similarity:
        print("No strong match found. The source collection may not cover this issue.")
        print(cosine_similarity(query_vector, document_vectors))
        return
    # this (below) creates a list of the indexes of the ranked similarity scores named ranked
    ranked = [i for i in scores.argsort()[::-1] # .argsort() returns the indexes of the items in the list if the itey were rearranged in ascending order based on item value
              if scores[i] >= max(min_similarity, 0.15 * scores.max())][:top_k] # essentially only allows indexes to be appended if they meat the max score of 0.15 or 15% of the max, then takes the top_k
    for rank, index in enumerate(ranked, start=1): # loops through the items in the list. enumerate gives an items position and its value - so rank is index in list ranked and index is the index of that value in the original document list
        reference, text, tags = records[index] # retrieves the tuple of information from the original records
        print(f"\n{rank}. {reference} | similarity: {scores[index]:.2f}") # prints the info
        print("SOURCE:", text) # prints actual source text


In [91]:
QUESTION = "What is the max portion of the grade that summative assessments can have?"
research(QUESTION)
# first runs: no strong match found. tried increasing min_similarity, tried changing question to grading policy * 5. had to lower min threshold a lot to get any results

(9, 3482)
(1, 3482)
(1, 9)
(9,)
QUESTION: What is the max portion of the grade that summative assessments can have?

1. Grading Framework | similarity: 0.23
SOURCE:  new york city public schools (nycps) stuyvesant high school 345 chambers street new york, ny 10282 grading framework school year 2026 – 2027 grades are an important way of communicating with our students; they provide feedback to them (and families) on their learning progress and their acquisition of content knowledge and skills. our mission at stuyvesant is “to instill the intellectual, moral and humanistic values necessary for each child to achieve their maximum potential as a student and as a caring citizen of the world.” therefore, our grading framework provides guidance for how we position and prepare our students to achieve our institutional goal. the stuyvesant grading framework includes three categories (with examples of each) which will contribute to a student’s final grade in a course. departments will determine 